# 02c: Train/Test Split

**Purpose:** Create stratified train/test splits and CV folds

**Dataset:** COMPAS (transformed)

**Date:** 2025-11-08

---

## Overview

### Purpose
- Create stratified 80/20 train/test split
- Generate 5-fold cross-validation indices
- Verify balance across splits
- Save split indices for reproducibility

### Outputs
- Train/test data → `data/processed/compas_train.parquet`, `compas_test.parquet`
- CV fold indices → `data/processed/cv_folds.json`
- Split metadata → `data/metadata/split_log.json`

### Runtime: <1 minute

---

In [1]:
# Setup
import sys
from pathlib import Path
import json
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split, StratifiedKFold

project_root = Path.cwd().parent.parent

PROCESSED_DIR = project_root / "data" / "processed"
METADATA_DIR = project_root / "data" / "metadata"

RANDOM_STATE = 42
TEST_SIZE = 0.2
N_FOLDS = 5

print("✓ Setup complete")

✓ Setup complete


## 1. Load Transformed Data

In [2]:
# Load transformed features
X = pd.read_parquet(PROCESSED_DIR / "compas_features_transformed.parquet")
y = pd.read_parquet(PROCESSED_DIR / "compas_target.parquet")['two_year_recid']
sensitive = pd.read_parquet(PROCESSED_DIR / "compas_sensitive.parquet")

print(f"Features: {X.shape}")
print(f"Target: {y.shape}")
print(f"Class distribution: {y.value_counts().to_dict()}")

Features: (5209, 9)
Target: (5209,)
Class distribution: {0: 2665, 1: 2544}


## 2. Train/Test Split

Stratified split to maintain class balance.

In [3]:
# Stratified train/test split
X_train, X_test, y_train, y_test, idx_train, idx_test = train_test_split(
    X, y, X.index,
    test_size=TEST_SIZE,
    stratify=y,
    random_state=RANDOM_STATE
)

print(f"Train set: {len(X_train):,} samples")
print(f"Test set: {len(X_test):,} samples")
print(f"\nTrain class distribution:")
print(y_train.value_counts())
print(f"\nTest class distribution:")
print(y_test.value_counts())

Train set: 4,167 samples
Test set: 1,042 samples

Train class distribution:
two_year_recid
0    2132
1    2035
Name: count, dtype: int64

Test class distribution:
two_year_recid
0    533
1    509
Name: count, dtype: int64


## 3. Verify Balance Across Splits

In [4]:
# Check class balance
train_positive_rate = y_train.mean()
test_positive_rate = y_test.mean()
overall_positive_rate = y.mean()

print(f"Positive class rates:")
print(f"  Overall: {overall_positive_rate:.1%}")
print(f"  Train:   {train_positive_rate:.1%}")
print(f"  Test:    {test_positive_rate:.1%}")
print(f"\nDifference: {abs(train_positive_rate - test_positive_rate):.2%}")

if abs(train_positive_rate - test_positive_rate) < 0.02:
    print("✓ Class balance maintained across splits")
else:
    print("⚠ Class imbalance detected between splits")

Positive class rates:
  Overall: 48.8%
  Train:   48.8%
  Test:    48.8%

Difference: 0.01%
✓ Class balance maintained across splits


## 4. Create CV Folds

In [6]:
# Create stratified k-fold splits for cross-validation
skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=RANDOM_STATE)

cv_folds = []
for fold_idx, (train_idx, val_idx) in enumerate(skf.split(X_train, y_train)):
    fold_info = {
        'fold': fold_idx,
        'train_indices': train_idx.tolist(),
        'val_indices': val_idx.tolist(),
        'train_size': len(train_idx),
        'val_size': len(val_idx),
        'train_positive_rate': float(y_train.iloc[train_idx].mean()),
        'val_positive_rate': float(y_train.iloc[val_idx].mean())
    }
    cv_folds.append(fold_info)
    
    print(f"Fold {fold_idx}: train={len(train_idx):,}, val={len(val_idx):,}, "
          f"train_pos={fold_info['train_positive_rate']:.1%}, "
          f"val_pos={fold_info['val_positive_rate']:.1%}")

print(f"\n✓ Created {N_FOLDS} CV folds")

Fold 0: train=3,333, val=834, train_pos=48.8%, val_pos=48.8%
Fold 1: train=3,333, val=834, train_pos=48.8%, val_pos=48.8%
Fold 2: train=3,334, val=833, train_pos=48.8%, val_pos=48.9%
Fold 3: train=3,334, val=833, train_pos=48.8%, val_pos=48.9%
Fold 4: train=3,334, val=833, train_pos=48.8%, val_pos=48.9%

✓ Created 5 CV folds


## 5. Save Splits

In [7]:
# Verify indices are disjoint (no data leakage)
train_idx_set = set(idx_train)
test_idx_set = set(idx_test)
overlap = train_idx_set.intersection(test_idx_set)

assert len(overlap) == 0, f"ERROR: {len(overlap)} overlapping indices between train and test!"
assert len(train_idx_set) + len(test_idx_set) == len(X), "ERROR: Index count mismatch!"

print(f"✓ Verified: train and test indices are disjoint")
print(f"  Train indices: {len(train_idx_set):,}")
print(f"  Test indices: {len(test_idx_set):,}")
print(f"  Total: {len(train_idx_set) + len(test_idx_set):,}")

✓ Verified: train and test indices are disjoint
  Train indices: 4,167
  Test indices: 1,042
  Total: 5,209


In [8]:
# Save train/test data (preserving indices to prevent data leakage)
X_train.to_parquet(PROCESSED_DIR / "compas_X_train.parquet", index=True)
X_test.to_parquet(PROCESSED_DIR / "compas_X_test.parquet", index=True)
y_train.to_frame().to_parquet(PROCESSED_DIR / "compas_y_train.parquet", index=True)
y_test.to_frame().to_parquet(PROCESSED_DIR / "compas_y_test.parquet", index=True)

# Save sensitive attributes for train/test
sensitive.loc[idx_train].to_parquet(PROCESSED_DIR / "compas_sensitive_train.parquet", index=True)
sensitive.loc[idx_test].to_parquet(PROCESSED_DIR / "compas_sensitive_test.parquet", index=True)

print("✓ Saved train/test splits")

# Save CV fold indices
with open(PROCESSED_DIR / "cv_folds.json", 'w') as f:
    json.dump(cv_folds, f, indent=2)
print("✓ Saved CV fold indices")

# Save split metadata
split_log = {
    'notebook': '02c_train_test_split.ipynb',
    'random_state': RANDOM_STATE,
    'test_size': TEST_SIZE,
    'n_folds': N_FOLDS,
    'total_samples': len(X),
    'train_samples': len(X_train),
    'test_samples': len(X_test),
    'class_balance': {
        'overall': float(overall_positive_rate),
        'train': float(train_positive_rate),
        'test': float(test_positive_rate)
    },
    'files': {
        'X_train': 'compas_X_train.parquet',
        'X_test': 'compas_X_test.parquet',
        'y_train': 'compas_y_train.parquet',
        'y_test': 'compas_y_test.parquet',
        'cv_folds': 'cv_folds.json'
    }
}

with open(METADATA_DIR / "split_log.json", 'w') as f:
    json.dump(split_log, f, indent=2)
print("✓ Saved split metadata")

✓ Saved train/test splits
✓ Saved CV fold indices
✓ Saved split metadata


## Summary

**Train/Test Split Complete:**
- ✓ 80/20 stratified split (maintains class balance)
- ✓ 5-fold cross-validation indices created
- ✓ All splits saved for reproducibility
- ✓ Sensitive attributes tracked separately

**Outputs:**
- compas_X_train.parquet, compas_X_test.parquet
- compas_y_train.parquet, compas_y_test.parquet
- compas_sensitive_train.parquet, compas_sensitive_test.parquet
- cv_folds.json (5 folds)
- split_log.json (metadata)

**Next:** 02d_preprocessing_validation.ipynb